# Results for model: nvidia/nemotron-mini-4b-instruct

In [ ]:
import polars as pl
import xgboost as xgb

# 1. Load train.parquet using Polars
train_df = pl.read_parquet('./jane_street_data/train.parquet')

# 2. Feature Engineering: Calculate a global TOP_QUANTILE of 'feature_00'
#   relative to 'responder_6' across rolling batches of 'date_id'
TOP_QUANTILE = 0.85
feature_00_q = train_df['feature_00'].quantile(TOP_QUANTILE, axis=0, kind='lower')
train_df['feature_00_rel'] = train_df['feature_00'] / feature_00_q

# 3. Train an XGBoost Regressor on the target 'responder_6'
xgb_reg = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='mae')
xgb_reg.fit(train_df[['date_id', 'feature_00_rel']], train_df['responder_6'].astype(float))

# Save the model for inference
xgb_reg.save_model('./jane_street_data/model.joblib')